In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage import io, color, exposure, filters, morphology, measure, util
from scipy import ndimage as ndi
from pdf2image import convert_from_path
import os
import glob
import seaborn as sns

In [2]:
experiment_dir = '.'

used_rows = ['A','B','C','D','E','F','G','H']
used_cols = ['1','2','3','4','5','6','7','8','9','10','11','12']

circle_area = 0.6
min_size = 20 # 5

In [3]:
def convert_pdf_to_pngs(pdf_path, analysis_dir, used_wells, crop_coords = (822, 446, 2478, 2102)):
    pages = convert_from_path(pdf_path, dpi=300)

    png_paths = []

    for well, page in zip(used_wells, pages):
        png_path = os.path.join(analysis_dir, f'{well}.png')
        cropped = page.crop(crop_coords)  # adjust coords once based on ELISPOT output
        cropped.save(png_path)
        png_paths.append((well, png_path))

    return png_paths

def load_gfp_images(png_paths):
    wells_raw = []

    for well, png_path in png_paths:
        img = io.imread(png_path)
        
        if img.ndim == 3:
            # take green
            green = img[:,:,1].astype(float)
        else:
            # convert to grayscale
            green = color.rgb2gray(img).astype(float)

        wells_raw.append((well, green))

    return wells_raw


def mask_images(wells_raw, circle_area):
    wells_masked = []
    
    for well, green in wells_raw:
        h, w = green.shape
    
        circle_radius_prop = circle_area**0.5
        radius = int(min(h, w)/2 * circle_radius_prop)
    
        mask = np.zeros_like(green, dtype=bool)  # set up empty mask
        cy, cx = h//2, w//2  # find the center of the tile
        Y, X = np.ogrid[:h, :w]  # coordinate grid
        circle = (X-cx)**2 + (Y-cy)**2 <= radius**2  # find pixels within the defined circle
        mask[circle] = True  # set circle region to True
        
        # apply the mask
        masked = green * mask
    
        wells_masked.append((well, masked, mask))

    return wells_masked

def get_gfp_threshold(wells_masked):
    all_maskeds = [t[1] for t in wells_masked]
    
    stacked_maskeds = np.hstack([masked.ravel() for masked in all_maskeds]) 

    thresh_val = filters.threshold_otsu(stacked_maskeds)
    
    return thresh_val

def analyze_gfp_regions(wells_masked, used_rows, used_cols, thresh_val, min_size):
    wells_binarized = []
    wells_props = []
    results = []
    
    for well, masked, mask in wells_masked:
        binarized = (masked > thresh_val) & mask
        binarized = morphology.remove_small_objects(binarized, min_size=min_size)
        binarized = ndi.binary_fill_holes(binarized)

        wells_binarized.append((well, binarized))

        gfp_area = np.sum(binarized)  # number of GFP(+) pixels
        well_area = np.sum(mask)  # total pixels in the circular mask
        frac_area = gfp_area / well_area if well_area > 0 else 0  # ratio
        
        results.append({
            "row": well[0],  # well row index
            "col": well[1:],  # well column index
            "well": well,
            "gfp_area_fraction": frac_area  # GFP area fraction
        })

    return wells_binarized, wells_props, results

def visualize_plate(wells_binarized, wells_masked, used_rows, used_cols, save_to = None, save_format = 'png', dpi=1000):
    n_rows = len(used_rows)
    n_cols = len(used_cols)
    
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols, n_rows), dpi=dpi, squeeze=False, facecolor="black")

    for (well, binarized), (_well, masked, mask) in zip(wells_binarized, wells_masked):
        if well != _well:
            print("WARNING: mismatched binarized and masked images")
            
        row_name = well[0]
        col_name = well[1:]
        
        r = used_rows.index(row_name)
        c = used_cols.index(col_name)

        ax = axs[r,c]  # subplot for this well
        ax.imshow(masked, cmap="gray", vmin=0, vmax=255)  # show masked grayscale image
        ax.contour(binarized, colors='lime', linewidths=0.125)  # outline GFP(+) regions
        ax.axis("off")  # remove axis ticks
        ax.set_aspect("equal")

    plt.tight_layout(pad=0)

    if save_to:
        plt.savefig(save_to, format=save_format)
        plt.close()
    else:
        plt.show()

In [4]:
pdf_paths = glob.glob(os.path.join(experiment_dir, '*', '*.pdf'))
analysis_dirs = [os.path.join(os.path.split(pdf_path)[0], 'analysis') for pdf_path in pdf_paths]

for pdf_path, analysis_dir in zip(pdf_paths, analysis_dirs):
    os.makedirs(analysis_dir, exist_ok = True)
    save_image_to = os.path.join(analysis_dir, 'plate_masked.png')
    save_csv_to = os.path.join(analysis_dir, 'gfp_results.csv')

    used_wells = [row_name+col_name for row_name in used_rows for col_name in used_cols]

    png_paths = convert_pdf_to_pngs(pdf_path, analysis_dir, used_wells)
    wells_raw = load_gfp_images(png_paths)
    wells_masked = mask_images(wells_raw, circle_area)
    thresh_val = get_gfp_threshold(wells_masked)
    wells_binarized, well_props, results = analyze_gfp_regions(wells_masked, used_rows, used_cols, thresh_val, min_size)
    visualize_plate(wells_binarized, wells_masked, used_rows, used_cols, save_to = save_image_to)

    pd.DataFrame(results).to_csv(save_csv_to)

    print(f'Finished analyzing {pdf_path}')

Finished analyzing ./example/example.pdf
